# ReelMatch — Prepare & Train

Filters the raw TMDB+IMDB dataset down to a lightweight, deployable subset,
builds a content-based recommendation model (TF-IDF + cosine similarity),
and saves compact artifacts for the Flask backend to load at startup.

Run all cells once before deployment.

## 1 — Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import re
import os
import pickle
import gzip
import pathlib
import shutil
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

In [ ]:
# --- Paths ---
# This notebook lives in  <project_root>/train/
# so the project root is one level up.
PROJECT_ROOT = pathlib.Path(os.getcwd()).resolve()
# If you run the notebook from inside the train/ folder, uncomment the next line:
# PROJECT_ROOT = pathlib.Path(os.getcwd()).resolve().parent

RAW_CSV   = PROJECT_ROOT / "TMDB  IMDB Movies Dataset.csv"
OUT_DIR   = PROJECT_ROOT / "backend" / "model_artifacts"

# --- Hyper-parameters ---
TOP_N_MOVIES   = 12_000   # keeps the final artifacts small & fast to load
MIN_VOTE_COUNT = 30       # drop obscure / low-signal entries
MAX_FEATURES   = 15_000   # cap TF-IDF vocabulary size

os.makedirs(OUT_DIR, exist_ok=True)
print(f"CSV path  : {RAW_CSV}")
print(f"Output dir: {OUT_DIR}")
print(f"CSV exists: {RAW_CSV.exists()}")

## 2 — Helper Functions

In [ ]:
def clean_text(x):
    if pd.isna(x):
        return ""
    return str(x)


def build_soup(row):
    """Combine relevant text fields into one 'soup' for TF-IDF."""
    genres    = clean_text(row["genres"]).replace(",", " ")
    keywords  = clean_text(row["keywords"]).replace(",", " ")
    overview  = clean_text(row["overview"])
    director  = clean_text(row["directors"]).replace(",", " ")
    cast      = clean_text(row["cast"]).replace(",", " ")
    # cast list can be long; keep first 5 to keep signal high & noise low
    cast_top5 = " ".join(cast.split(", ")[:5]) if cast else ""
    tagline   = clean_text(row["tagline"])

    soup = f"{overview} {tagline} {genres} {genres} {keywords} {director} {director} {cast_top5}"
    soup = re.sub(r"[^a-zA-Z0-9\s]", " ", soup).lower()
    soup = re.sub(r"\s+", " ", soup).strip()
    return soup

## 3 — Load & Filter the Dataset

In [ ]:
print("Loading raw CSV (this may take a minute)...")
usecols = [
    "id", "title", "vote_average", "vote_count", "status", "release_date",
    "runtime", "adult", "original_language", "overview", "popularity",
    "poster_path", "tagline", "genres", "keywords", "directors", "cast",
]
df = pd.read_csv(RAW_CSV, usecols=usecols, low_memory=False)
print(f"Loaded {len(df)} rows")
df.head()

In [ ]:
# --- Filtering to keep the deployed model lightweight & high quality ---
df = df[df["status"] == "Released"]
df = df[df["adult"] == False]
df = df[df["poster_path"].notna()]
df = df[df["overview"].notna() & (df["overview"].str.len() > 20)]
df = df[df["vote_count"].fillna(0) >= MIN_VOTE_COUNT]
df = df.drop_duplicates(subset=["title", "release_date"])

# Rank by a blend of popularity and vote_count, take top N
df["popularity"]  = df["popularity"].fillna(0)
df["vote_count"]  = df["vote_count"].fillna(0)
df["score_rank"]  = df["popularity"].rank(pct=True) * 0.6 + df["vote_count"].rank(pct=True) * 0.4
df = df.sort_values("score_rank", ascending=False).head(TOP_N_MOVIES)
df = df.reset_index(drop=True)
print(f"Kept {len(df)} movies after filtering")

## 4 — Build Text Features (Soup)

In [ ]:
print("Building text features...")
df["soup"] = df.apply(build_soup, axis=1)
df[["title", "soup"]].head()

## 5 — TF-IDF + Nearest Neighbors

In [ ]:
print("Fitting TF-IDF vectorizer...")
tfidf = TfidfVectorizer(max_features=MAX_FEATURES, stop_words="english")
tfidf_matrix = tfidf.fit_transform(df["soup"])
print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")

In [ ]:
print("Fitting Nearest Neighbors index...")
nn_model = NearestNeighbors(n_neighbors=11, metric="cosine", algorithm="brute")
nn_model.fit(tfidf_matrix)
print("Done!")

## 6 — Prepare Metadata Table

In [ ]:
df["release_year"] = pd.to_datetime(df["release_date"], errors="coerce").dt.year
df["vote_average"] = df["vote_average"].fillna(0).round(1)
df["poster_url"]   = df["poster_path"].apply(
    lambda p: f"https://image.tmdb.org/t/p/w500{p}" if pd.notna(p) else None
)

meta = df[[
    "id", "title", "overview", "vote_average", "vote_count",
    "release_year", "poster_url", "genres", "runtime",
]].copy()
meta["title_lower"] = meta["title"].str.lower().str.strip()
meta.head()

## 7 — Save Artifacts

In [ ]:
print("Saving artifacts...")

with gzip.open(str(OUT_DIR / "tfidf_matrix.pkl.gz"), "wb") as f:
    pickle.dump(tfidf_matrix, f, protocol=pickle.HIGHEST_PROTOCOL)

with gzip.open(str(OUT_DIR / "nn_model.pkl.gz"), "wb") as f:
    pickle.dump(nn_model, f, protocol=pickle.HIGHEST_PROTOCOL)

with gzip.open(str(OUT_DIR / "tfidf_vectorizer.pkl.gz"), "wb") as f:
    pickle.dump(tfidf, f, protocol=pickle.HIGHEST_PROTOCOL)

meta.to_pickle(str(OUT_DIR / "movies_meta.pkl.gz"), compression="gzip")

print("\nDone! Artifact files:")
for fname in os.listdir(OUT_DIR):
    fpath = OUT_DIR / fname
    size_mb = fpath.stat().st_size / (1024 * 1024)
    print(f"  {fname:40s} {size_mb:.2f} MB")

## 8 — Quick Sanity Check

Test the model with a sample movie to verify everything works before starting the Flask app.

In [ ]:
# Quick test: find movies similar to "Inception"
test_title = "inception"
title_to_index = {t: i for i, t in enumerate(meta["title_lower"])}

if test_title in title_to_index:
    idx = title_to_index[test_title]
    distances, indices = nn_model.kneighbors(tfidf_matrix[idx], n_neighbors=11)
    print(f"Movies similar to '{meta.iloc[idx]['title']}':\n")
    for dist, rec_idx in zip(distances[0], indices[0]):
        if rec_idx == idx:
            continue
        similarity = 1 - dist
        print(f"  {similarity:.4f}  {meta.iloc[rec_idx]['title']} ({meta.iloc[rec_idx]['release_year']})")
else:
    print(f"'{test_title}' not found in the dataset. Try another title.")